In [47]:
import ast
import json
from glob import glob

import pandas as pd
import os

In [48]:
BASE = "/Users/charles-/Downloads/Master/M2/PersonasData/Stupeflip"
CURRENT_ARTIST = "Stupeflip"
SONGS_EXT = "tracks/songsWithRhymeSchemes/*.json"

In [49]:
PERSONAS = ["kingju", "pophip"] # identifiants de persona qu'on veut donner à chacune

## Création persona_attribution_persona.csv

In [50]:
# writer_artist_ids = ["[820]", "[820]", "[4389143]"] # à chercher avec un tracksDfs[i] > writer_artists_id. 

In [52]:
personaFolders = [os.path.join(BASE, persona) for persona in PERSONAS]
tracksDfsPaths = [os.path.join(persona_folder, "tracksDf.csv") for persona_folder in personaFolders]
tracksDfs = [pd.read_csv(path) for path in tracksDfsPaths]

In [53]:
# # Station de nettoyage des featurings. Non obligatoire si vous êtes moins des sauvages que moi et que vous avez trié à la main
# tracksDfs = [
#     df[df["writer_artists_id"] == writer_id]
#     for df, writer_id in zip(tracksDfs, writer_artist_ids)

In [30]:
persona_attribution_dfs = [
    pd.DataFrame({
        "id": df["song_id"],
        "title": df["title"],
        #"album_id": df["album_id"],
        "persona_id": persona,  # scalar is broadcast, no need for [persona]*l
    })
    for persona, df in zip(PERSONAS, tracksDfs)
]

In [31]:
persona_attribution_df = pd.concat(persona_attribution_dfs)

In [60]:
persona_attribution_df.to_csv(f"{CURRENT_ARTIST}_attribution_persona.csv")

# Ajout persona dans les fichiers de chansons

In [61]:
def add_persona_id(sections, persona_id):
    for section in sections:
        for line in section.get("lines", []):
            line["persona_id"] = persona_id
    return sections


def load_and_apply_persona(path_pattern, ids, persona_id):
    """Charge les JSON, garde les ids voulus et ajoute la persona à chaque ligne."""
    rows = []
    for path in glob(path_pattern):
        with open(path, encoding="utf-8") as f:
            song = json.load(f)
        if song["song_id"] in ids:
            song["lyrics"] = add_persona_id(song["lyrics"], persona_id)
            rows.append(song)
    return pd.DataFrame(rows)

In [62]:
attribution_df = pd.read_csv(f"{CURRENT_ARTIST}_attribution_persona.csv").drop("Unnamed: 0", axis=1)

In [64]:
dfs = list()
for persona in PERSONAS:
    path_to_persona = os.path.join(BASE, persona)
    path_to_songs = os.path.join(path_to_persona, SONGS_EXT)
    songs_ids_persona = attribution_df[attribution_df["persona_id"] == persona]["id"].to_list()
    df_persona = load_and_apply_persona(path_to_songs, songs_ids_persona, persona_id=persona)
    dfs.append(df_persona)

artist_df = pd.concat(dfs)

In [66]:
# Sauvegarde : lyrics sérialisé en vrai JSON
artist_df.assign(lyrics=artist_df["lyrics"].apply(json.dumps, ensure_ascii=False)).to_csv(f"{BASE}/artist_df.csv", index=False)

# Passage du dataframe artiste à des json par chanson pour mesures

In [72]:
artist_df

,song_id,title,language,lyricists,lyrics
0,2969121,Tales from the Crou,fr,[55334],"[{'section_id': 'section_1', 'type': 'intro', ..."
1,10953234,Déjà tout petit (1995 c.t.),fr,[55334],"[{'section_id': 'section_0', 'type': 'couplet'..."
2,11038625,Stup Virus (Flip Party version),fr,[55334],"[{'section_id': 'section_1', 'type': 'couplet'..."
3,2965381,The Antidote,en,[55334],"[{'section_id': 'section_1', 'type': 'intro', ..."
4,2968663,Grosse tête,NaN,[55334],[]
...,...,...,...,...,...
9,7362715,La Formule Secrète,fr,[1019559],"[{'section_id': 'section_0', 'type': 'intro', ..."
10,7112176,Gaëlle (refrain inédit),fr,[1019559],"[{'section_id': 'section_0', 'type': 'couplet'..."
11,243042,La bavure de Pop Hip,en,[1019559],"[{'section_id': 'section_0', 'type': 'couplet'..."
12,242242,Carry On,en,[1019559],"[{'section_id': 'section_0', 'type': 'couplet'..."


In [91]:
def get_persona(lyrics):
    for section in lyrics:
        for line in section.get("lines", []):
            if line.get("persona_id"):
                return line["persona_id"]
    return None

artist_df["persona"] = artist_df["lyrics"].apply(get_persona)

for sid, group in artist_df.groupby("song_id"):
    if sid in pophip_songs:
        match = group[group["persona"] == "pophip"]
        row = match.iloc[0] if not match.empty else group.iloc[0]
    else:
        row = group.iloc[0]
    row.drop("persona").to_json(
        f"Corpus/{CURRENT_ARTIST}_songs/{sid}.json",
        force_ascii=False,
        indent=2,
    )

In [92]:
pophip_songs = {
    10953224, 7994787, 2968643, 330651, 343884, 226166, 382103,
    241969, 242242, 243042, 243099, 380555, 7362715, 7112176,
}

# tests

In [93]:
import json
from pathlib import Path

def get_file_persona(json_path):
    """Retourne le persona_id d'un fichier (le premier trouvé), ou None."""
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    for section in data.get("lyrics", []):
        for line in section.get("lines", []):
            if line.get("persona_id"):
                return line["persona_id"]
    return None

def count_files_by_persona(folder, persona="pophip"):
    folder = Path(folder)
    files = sorted(folder.glob("*.json"))
    matching = [p for p in files if get_file_persona(p) == persona]
    print(f"{len(matching)} fichier(s) '{persona}' sur {len(files)} dans {folder}")
    return matching

# Dossier du fichier uploadé : à adapter à ton chemin réel
matching = count_files_by_persona("Corpus/Stupeflip_songs", "pophip")

14 fichier(s) 'pophip' sur 169 dans Corpus/Stupeflip_songs


In [94]:
from collections import Counter

def persona_counts(folder):
    return Counter(get_file_persona(p) for p in Path(folder).glob("*.json"))

print(persona_counts("Corpus/Stupeflip_songs"))

Counter({'kingju': 127, None: 28, 'pophip': 14})


In [95]:
import json
from pathlib import Path

def list_none_files(folder):
    """Liste les fichiers dont get_file_persona renvoie None, avec la raison."""
    none_files = []
    for p in sorted(Path(folder).glob("*.json")):
        try:
            with open(p, encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            none_files.append((p.name, f"JSON illisible ({type(e).__name__})"))
            continue

        sections = data.get("lyrics") or []
        lines = [l for s in sections for l in s.get("lines", [])]

        if not sections:
            reason = "pas de section 'lyrics'"
        elif not lines:
            reason = "aucune ligne dans les sections"
        elif not any(l.get("persona_id") for l in lines):
            reason = "aucun 'persona_id' (absent ou vide) sur les lignes"
        else:
            continue  # persona trouvé, fichier OK

        none_files.append((p.name, reason))

    print(f"{len(none_files)} fichier(s) sans persona :")
    for name, reason in none_files:
        print(f"  - {name} : {reason}")
    return none_files

none_files = list_none_files("Corpus/Stupeflip_songs")

28 fichier(s) sans persona :
  - 10953222.json : pas de section 'lyrics'
  - 10953232.json : pas de section 'lyrics'
  - 10953240.json : pas de section 'lyrics'
  - 10953241.json : pas de section 'lyrics'
  - 12539208.json : pas de section 'lyrics'
  - 1845662.json : pas de section 'lyrics'
  - 2086801.json : pas de section 'lyrics'
  - 241363.json : pas de section 'lyrics'
  - 251736.json : pas de section 'lyrics'
  - 251741.json : pas de section 'lyrics'
  - 2938098.json : pas de section 'lyrics'
  - 2938105.json : aucune ligne dans les sections
  - 2938106.json : aucune ligne dans les sections
  - 2968633.json : pas de section 'lyrics'
  - 2968663.json : pas de section 'lyrics'
  - 2969120.json : pas de section 'lyrics'
  - 2975023.json : pas de section 'lyrics'
  - 3009127.json : pas de section 'lyrics'
  - 686726.json : pas de section 'lyrics'
  - 7352649.json : pas de section 'lyrics'
  - 7994777.json : pas de section 'lyrics'
  - 7994778.json : pas de section 'lyrics'
  - 799478